# MobApp Studio, workshop 2 — Charts that do not lie

John asked for six specific charts. You can produce all six in twenty minutes with pandas, and
**two of them will be misleading** unless you notice what is hiding in the data first.

That is the uncomfortable part of this job: the brief tells you what to plot, and it is still your
name on the slide. Today:

1. **A chart the brief asked for that is wrong** — the mean price per category.
2. **When a linear axis hides everything** — installs and reviews are power laws.
3. **What a pie chart cannot show**, and the chart John actually needs instead.
4. **Numbers that are not measurements** — `Installs` has only twenty possible values.

In [ ]:
# ABOUTME: MobApp Studio workshop 2 - making the brief's charts, and noticing when they mislead.
# ABOUTME: Troll apps in the mean, log scales for power laws, and install buckets.
import matplotlib.pyplot as plt

from pathlib import Path
import numpy as np
import pandas as pd

def qwasar_file(name):
    """Locate a dataset fetched by workshops/get_data.py, from wherever this notebook was opened."""
    for up in range(4):
        p = Path(*([".."] * up), "data", name)
        if p.exists():
            return p
    raise FileNotFoundError(
        f"{name} is missing. Run `uv run get_data.py` in the workshops/ folder first - "
        "these notebooks only ever use the real dataset the project links to.")

# The Google Play export the project points at. 10,841 rows, exactly as published.
raw = pd.read_csv(qwasar_file("googleplaystore.csv"))

pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 20)
print(f"loaded {len(raw):,} rows x {raw.shape[1]} columns from the real Google Play export")


In [ ]:
# The cleaning from workshop 1, in one cell, so this notebook stands alone.
df = raw[raw.Category.astype(str).str.match(r"^[A-Z_]+$")].copy()
df["Reviews"] = pd.to_numeric(df.Reviews, errors="coerce")
df = df.sort_values("Reviews", ascending=False).drop_duplicates(subset="App").reset_index(drop=True)
df["Installs"] = pd.to_numeric(df.Installs.astype(str).str.replace(r"[,+]", "", regex=True), errors="coerce")
df["Price"] = pd.to_numeric(df.Price.astype(str).str.replace("$", "", regex=False), errors="coerce")
df["Paid"] = df.Price > 0
print(f"{len(df):,} apps, {int(df.Paid.sum()):,} of them paid")

## 1. The chart that lies

> *"Plot a bar with the mean price per category."* — the brief

One `groupby`, one `.mean()`, one bar chart. Here it is.

In [ ]:
mean_price = df.groupby("Category").Price.mean().nlargest(10)
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh([c.title() for c in mean_price.index][::-1], mean_price.values[::-1], color="#c44")
ax.set_xlabel("mean price ($)"); ax.set_title("Mean price per category - as requested")
plt.tight_layout()

Finance apps average several dollars while everything else sits near zero. A VP would read that as
*"finance is where the money is"* and plan a budget around it.

Now look at what is actually in that column.

### Exercise 1 — find out who is holding up the average

In [ ]:
# TODO 1: list the apps priced above $100 (App, Category, Price)
# TODO 2: for FINANCE, print the mean price and the median price, over all apps and paid-only
expensive = None
print(expensive)

In [ ]:
def check(label, *conditions):
    """Report an exercise self-check without raising: a red traceback is a bad first impression."""
    try:
        ok = all(bool(c()) for c in conditions)
    except Exception as exc:                      # a stub that still returns None lands here
        print(f"  {label}: not yet ({type(exc).__name__}: {exc})")
        return False
    print(f"  {label}: {'PASS' if ok else 'not yet'}")
    return ok

check("found the expensive apps", lambda: expensive is not None and len(expensive) > 5)
check("they are joke apps", lambda: expensive.App.str.contains("rich", case=False).any())

**`I am rich`. `I Am Rich Premium`. `I'm Rich - Trump Edition`.** Twenty apps whose entire function
is to display a glowing gem to prove you wasted $400. They are a running joke on the store, and
they are sitting in your chart as though they were a market segment.

The numbers for FINANCE: a mean of **$8.41** against a median of **$0.00**. Among paid apps only,
a mean of **$170.64** against a median of **$28.99**. The mean is not describing the typical app in
any of those cuts; it is describing a handful of pranks.

**Rule of thumb:** a mean is only a summary when the distribution is roughly symmetric. Check the
median next to it; when they disagree wildly, the mean is reporting outliers, not the typical case.

### Exercise 2 — plot it honestly

Three fixes, and the brief is still satisfied:

1. Average **paid apps only** — including thousands of zeros answers a different question.
2. Show the **median** beside the mean, so the reader can see the skew.
3. Exclude the joke apps, **and say you did, with the count**.

In [ ]:
real_apps = None     # TODO 1: paid apps, excluding the ones priced over $100
summary = None       # TODO 2: per category, the mean and the median price, categories with >= 10 paid apps
# TODO 3: a grouped bar chart of the two, top 10 by mean
summary

In [ ]:
def check(label, *conditions):
    """Report an exercise self-check without raising: a red traceback is a bad first impression."""
    try:
        ok = all(bool(c()) for c in conditions)
    except Exception as exc:                      # a stub that still returns None lands here
        print(f"  {label}: not yet ({type(exc).__name__}: {exc})")
        return False
    print(f"  {label}: {'PASS' if ok else 'not yet'}")
    return ok

check("joke apps excluded", lambda: real_apps.Price.max() <= 100)
check("free apps excluded", lambda: (real_apps.Price > 0).all())
check("both statistics reported", lambda: {"mean", "median"} <= set(summary.columns))

Same data, same question, a chart that survives a VP asking "what is a typical app in that
category worth?"

## 2. The axis that hides everything

> *"Plot with a bar diagram the most populars paid apps of Family category."*

Installs span six orders of magnitude. On a linear axis, the biggest app flattens every other bar
into a smear at zero.

In [ ]:
fam_paid = df[(df.Category == "FAMILY") & df.Paid].nlargest(10, "Installs")

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for ax, scale in zip(axes, ["linear", "log"]):
    ax.barh([a[:26] for a in fam_paid.App][::-1], fam_paid.Installs.values[::-1], color="#36c")
    ax.set_xscale(scale); ax.set_xlabel("installs"); ax.set_title(f"{scale} axis")
plt.tight_layout()

Left: one app and nine stubs. Right: you can read every bar and compare them.

The same applies to the scatter matrix the project asks for. `Reviews` and `Installs` are power
laws — most apps have almost nothing, a handful have hundreds of millions — so on linear axes every
point lands in the bottom-left corner and the plot shows you nothing at all.

**Rule of thumb:** if the largest value is more than ~100x the median, plot the logarithm. Say so
on the axis label so nobody misreads the scale.

### Exercise 3 — how skewed is it, numerically?

Before reaching for a log axis, justify it with a number your report can quote.

In [ ]:
# TODO: for Installs and Reviews, compute the median, the max, and the ratio max/median
skew_table = None
skew_table

In [ ]:
def check(label, *conditions):
    """Report an exercise self-check without raising: a red traceback is a bad first impression."""
    try:
        ok = all(bool(c()) for c in conditions)
    except Exception as exc:                      # a stub that still returns None lands here
        print(f"  {label}: not yet ({type(exc).__name__}: {exc})")
        return False
    print(f"  {label}: {'PASS' if ok else 'not yet'}")
    return ok

check("both columns measured", lambda: set(skew_table.index) == {"Installs", "Reviews"})
check("the skew is enormous", lambda: skew_table["max / median"].min() > 100)

## 3. Numbers that are not measurements

> *"An array with the number of installation per category."*

Before summing `Installs`, look at what the column can contain.

In [ ]:
levels = sorted(df.Installs.dropna().unique())
print(f"{df.Installs.nunique()} distinct values for {len(df):,} apps:")
print([f"{v:,.0f}" for v in levels])
top = df.Installs.max()
print(f"\napps sitting at the top level ({top:,.0f}+): {int((df.Installs == top).sum())}")

Twenty levels. The store never published exact counts — it published **buckets**, and the raw value
was already a label like `"1,000,000+"`.

Two consequences you must carry into the report:

- Every total is a **lower bound**. An app in the `1,000,000,000+` bucket might have five billion
  installs; the data cannot tell you.
- The *precision* of your headline is fake. "75,322,526,427 installs" has eleven significant digits
  and maybe one real one. Write **"roughly 75 billion"**.

**Rule of thumb:** report numbers to the precision your data actually carries. Spurious digits are
a claim about accuracy you cannot back.

## 4. The chart John actually needs

> *"Depending on each category, what are the ratio of download per app?"*

This is the sharpest question in the brief, and the one a pie chart of totals cannot answer. A
category can be huge because it is popular, or merely because it is crowded.

### Exercise 4 — totals against ratios

In [ ]:
# TODO: per category, compute the number of apps, total installs, and installs per app.
#       Then show the top 8 by TOTAL next to the top 8 by RATIO (keep categories with >= 50 apps).
by_cat = None
by_cat

In [ ]:
def check(label, *conditions):
    """Report an exercise self-check without raising: a red traceback is a bad first impression."""
    try:
        ok = all(bool(c()) for c in conditions)
    except Exception as exc:                      # a stub that still returns None lands here
        print(f"  {label}: not yet ({type(exc).__name__}: {exc})")
        return False
    print(f"  {label}: {'PASS' if ok else 'not yet'}")
    return ok

check("ratio computed", lambda: "installs_per_app" in by_cat.columns)
check("shares add up to 100%", lambda: abs(by_cat["share_%"].sum() - 100) < 1e-6)
check("the two rankings differ", lambda: by_cat.nlargest(1, "installs").index[0]
                                        != by_cat[by_cat.apps >= 50].nlargest(1, "installs_per_app").index[0])

In [ ]:
if by_cat is None:
    print("Finish exercise 4, then run this cell to see the two views side by side.")
else:
  big = by_cat[by_cat.apps >= 50]
  fig, axes = plt.subplots(1, 2, figsize=(14, 5))
  top8 = by_cat.installs.nlargest(8)
  axes[0].pie(list(top8) + [by_cat.installs.sum() - top8.sum()],
              labels=[c.title() for c in top8.index] + ["Other"], autopct="%1.0f%%", startangle=90)
  axes[0].set_title("Share of installs (the crowd-pleaser)")
  ratio = big.installs_per_app.nlargest(8)
  axes[1].barh([c.title() for c in ratio.index][::-1], ratio.values[::-1], color="#4a7")
  axes[1].set_xlabel("installs per app"); axes[1].set_title("Installs per app (the business answer)")
  plt.tight_layout()

Read the two together and the recommendation changes.

**GAME** wins the pie at ~18 % of all installs — and it holds 945 apps, so the average game gets
about **14 million** installs. **COMMUNICATION** takes a smaller slice of the pie with only 315
apps, and the average app there gets **35 million** — two and a half times better.

If John's question is *"where will our one app get the most downloads?"*, the pie chart points at
the wrong category. Totals describe the market; ratios describe your odds in it.

**The punchline:** the brief asked for the pie. Your job is to deliver the pie **and** the chart
that answers the question behind it.

## What to take away

1. **Check the mean against the median** before plotting an average. Twenty joke apps moved a
   whole category to the top of a chart.
2. **Say what you excluded, and how many.** "20 novelty apps above $100 removed" is a sentence that
   protects you; a silently filtered chart is one someone else will catch.
3. **Log axes for power laws.** If max/median is in the hundreds, a linear axis is a blank chart.
4. **Match your precision to your data.** Bucketed installs do not justify eleven digits.
5. **Totals and ratios answer different questions.** Produce both; the ratio is usually the one the
   business actually asked for.

### Take it home
Produce the six charts the brief lists, and for each one write the single sentence you would say
while showing it. If a sentence needs a caveat ("excluding novelty apps", "note the log scale"),
put the caveat on the chart itself — that is what `print_histograms` and `print_scatter_matrix`
are really for.